# wandb-finish — worked example 2: Guarantee wandb.finish with try/finally

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-finish`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Placing `wandb.finish()` in a `try/finally` block ensures the run is closed even if training raises an exception. Without this, a crashed run stays open on wandb's servers (shown as 'running' indefinitely) and the next `wandb.init()` in a sweep may behave unexpectedly. The `finally` clause runs regardless of whether the `try` block completes normally or raises.

## Worked solution

**Step 1 — why a bare finish() is fragile.**
If you write `loss = compute_loss(...); wandb.finish()` and `compute_loss` raises `RuntimeError`, Python never reaches `wandb.finish()`. The run is left open. In a 100-trial sweep, a single crash can orphan the run and confuse the sweep dashboard.

**Step 2 — the try/finally pattern.**
Instead, we write `try: <training loop> finally: wandb.finish()`. Python guarantees that the `finally` block executes whether or not the `try` block raises. The exception still propagates after `finally` completes.

**Step 3 — verify both paths.**
We test that `wandb.finish()` is called when training completes normally (no raise), and also when training raises partway through. Both paths should result in exactly one `finish()` call.

In [ ]:
import sys
from unittest.mock import MagicMock, patch
sys.modules.setdefault('wandb', MagicMock())
import wandb

def safe_train(project, n_steps, fail_at=None):
    """Training loop that always calls wandb.finish, even on exception."""
    wandb.init(project=project, name='safe-run')
    completed = 0
    try:
        for step in range(n_steps):
            if fail_at is not None and step == fail_at:
                raise RuntimeError(f'Crash at step {step}')
            completed += 1
    finally:
        # Always runs — whether training succeeded or crashed
        wandb.finish()
    return completed

# Path 1: normal completion
wandb.init.reset_mock()
wandb.finish.reset_mock()
result = safe_train('proj', 5)
print('Normal path — steps completed:', result)
print('finish called:', wandb.finish.call_count, 'time(s)')

# Path 2: exception path
wandb.init.reset_mock()
wandb.finish.reset_mock()
try:
    safe_train('proj', 5, fail_at=2)
except RuntimeError as e:
    print('Exception path — caught:', e)
    print('finish called anyway:', wandb.finish.call_count, 'time(s)')